In [21]:
from bs4 import BeautifulSoup
import requests
import os 
import smtplib
import time
import datetime


In [26]:
def check_prices():
  headers = {
      "User-Agent": (
          "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML,"
          " like Gecko) Chrome/120.0.0.0 Safari/537.36"
      ),
      "Accept-Language": "en-US,en;q=0.9",
      "Accept": (
          "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8"
      ),
  }

  today = datetime.date.today()
  file_path = (
      r"C:\Users\lg\OneDrive\Belgeler\Desktop\amazon\AmazonWebScraperDataset.csv"
  )
  file_exists = os.path.isfile(file_path)

  total_new_rows = 0

  # "a+" mode use kar rahe hain taake purana data delete na ho, balki naya end mein add ho
  with open(file_path, "a+", newline="", encoding="UTF8") as f:
    writer = csv.writer(f)
    if not file_exists:
      writer.writerow(["Title", "Price", "Date"])

    # Yahan range ko 1 se 6 kar diya hai taake pehle 5 pages (1, 2, 3, 4, 5) scrape hon
    for page_num in range(1, 6):
      URL = f"https://www.amazon.com/s?k=data%2Banalyst%2Bshirt&page={page_num}&crid=1HZZP0Z752E2E&sprefix=data%2Banalyst%2Bshir%2Caps%2C432&ref=sr_pg_{page_num}"
      print(f"Scraping Page {page_num}...")

      try:
        page = requests.get(URL, headers=headers)
        page.raise_for_status()
      except requests.exceptions.RequestException as e:
        print(f"Page {page_num} par connection ka masla hai: {e}")
        continue

      soup = BeautifulSoup(page.content, "html.parser")
      products = soup.find_all("div", {"data-component-type": "s-search-result"})

      for product in products:
        try:
          title_elem = product.find("h2", class_="a-size-base-plus")
          if not title_elem:
            title_elem = product.find("span", class_="a-text-normal")
          title = title_elem.get_text().strip() if title_elem else "N/A"

          price_parent = product.find("span", class_="a-price")
          if price_parent:
            whole_price = price_parent.find("span", class_="a-price-whole")
            price = whole_price.get_text().strip() if whole_price else "N/A"
          else:
            price = "N/A"

          if title != "N/A" and price != "N/A" and len(title) > 5:
            writer.writerow([title, price, today])
            total_new_rows += 1

        except Exception:
          continue

      # Amazon block se bachne ke liye har page ke darmiyan 3 seconds ka gap
      time.sleep(3)

  print(f"Successfully scraped and appended {total_new_rows} rows!")


# Function call
check_prices()

# --- Duplicate Handling & Pandas Display Part ---
file_path = (
    r"C:\Users\lg\OneDrive\Belgeler\Desktop\amazon\AmazonWebScraperDataset.csv"
)

if os.path.isfile(file_path):
  df = pd.read_csv(file_path)
  if not df.empty:
    # Duplicate rows ko remove karna (agar title aur price same hon)
    df.drop_duplicates(subset=["Title", "Price"], keep="first", inplace=True)

    # Clean price column
    df["Price"] = (
        df["Price"].astype(str).str.replace(r"\n", "", regex=True).str.strip()
    )
    df = df[["Title", "Price", "Date"]]

    # Saaf ki hui unique file ko wapas save kar dena taake file mein bhi duplicates remove ho jayein
    df.to_csv(file_path, index=False)

    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 1000)
    pd.set_option("display.max_colwidth", 40)

    print(
        "\n--- Final Cleaned & Updated Dataset Preview (Duplicates Handled) ---"
    )
    print(df)
  else:
    print("CSV file khali hai.")

Scraping Page 1...
Scraping Page 2...
Scraping Page 3...
Scraping Page 4...
Scraping Page 5...
Successfully scraped and appended 37 rows!

--- Final Cleaned & Updated Dataset Preview (Duplicates Handled) ---
                                      Title   Price        Date
0   CHOORO Data Analyst Gift Data Scienc...  3,859.  2026-07-21
1   Data Analyst T Shirt Funny Gift for ...  8,883.  2026-07-21
2   Funny Shirts Best Data Analyst Ever ...  4,173.  2026-07-21
3   Funny Tee Shirts Best Data Analyst E...  4,109.  2026-07-21
4   Funny T-Shirt Best Data Analyst Ever...  4,157.  2026-07-21
5   Womens Funny Tshirts Best Data Analy...  4,157.  2026-07-21
6   T Shirts for Men Funny Best Data Ana...  4,187.  2026-07-21
7   T-Shirts for Men Graphic Funny Best ...  4,262.  2026-07-21
8   Tshirt Funny Best Data Analyst Ever ...  5,335.  2026-07-21
9   Debugging Expert Shirt Computer Code...  7,216.  2026-07-21
10  I'm 99 Percent Sure It's A False Pos...  5,279.  2026-07-21
11  Chartered Financial 

In [27]:
%%writefile app.py
import streamlit as st
import pandas as pd
import datetime
import os

# --- Streamlit Page Configuration ---
st.set_page_config(
    page_title="E-Commerce Niche Price Scraper",
    page_icon="🛍️",
    layout="wide"
)

file_path = r"C:\Users\lg\OneDrive\Belgeler\Desktop\amazon\AmazonWebScraperDataset.csv"

# --- App UI Title & Description ---
st.title("🛍️ Automated E-Commerce Competitor Price Tracker")
st.markdown("This dashboard displays your targeted Amazon niche scraped dataset, complete with duplicate handling and historical append logs.")

# --- Display Data & Analytics Section ---
if os.path.isfile(file_path):
    df_display = pd.read_csv(file_path)
    if not df_display.empty:
        # Metrics Row
        col1, col2, col3 = st.columns(3)
        col1.metric("Total Records in DB", len(df_display))
        col2.metric("Unique Products", df_display['Title'].nunique())
        col3.metric("Latest Update Date", str(datetime.date.today()))
        
        # Search filter inside table view
        search_term = st.text_input("Filter results by title keyword:")
        if search_term:
            df_display = df_display[df_display['Title'].str.contains(search_term, case=False, na=False)]
            
        # Display interactive dataframe table
        st.dataframe(df_display, use_container_width=True)
        
        # Download Button for CSV
        csv_data = df_display.to_csv(index=False).encode('utf-8')
        st.download_button(
            label="📥 Download Cleaned Dataset as CSV",
            data=csv_data,
            file_name="Amazon_Scraped_Dataset.csv",
            mime="text/csv"
        )
    else:
        st.info("The dataset file is currently empty. Run your scraping cell first!")
else:
    st.warning("No dataset found at the specified path. Please run your scraper code first to generate the CSV file.")

Writing app.py
